# V05 — Plotly Subplots & Multi-Panel Dashboards

**`make_subplots` is where individual charts become dashboards.** This notebook covers shared axes, mixed chart types in one figure, inset charts, and complex grid layouts — the skills needed to build production analytical reports in Plotly alone.

**Reference:** [Subplots docs](https://plotly.com/python/subplots/)

**Allowed:** `plotly.graph_objects`, `plotly.express`, `plotly.subplots`, `pandas`, `numpy`


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.datasets import fetch_openml, fetch_california_housing

# --- Datasets ---
retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['Month'] = retail['InvoiceDate'].dt.to_period('M').astype(str)
retail['DayOfWeek'] = retail['InvoiceDate'].dt.day_name()
retail['Hour'] = retail['InvoiceDate'].dt.hour
retail['CustomerID'] = retail['CustomerID'].astype(int)

monthly = retail.groupby('Month').agg(
    Revenue=('Revenue','sum'),
    Orders=('InvoiceNo','nunique'),
    Customers=('CustomerID','nunique')
).reset_index()
monthly['AvgOrderValue'] = monthly['Revenue'] / monthly['Orders']

top_countries = (
    retail.groupby('Country')
    .agg(Revenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'))
    .reset_index().nlargest(10, 'Revenue')
)

credit_raw = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto').frame
credit = credit_raw.copy()
credit['credit_amount'] = pd.to_numeric(credit['credit_amount'], errors='coerce')
credit['duration'] = pd.to_numeric(credit['duration'], errors='coerce')
credit['age'] = pd.to_numeric(credit['age'], errors='coerce')

housing_raw = fetch_california_housing(as_frame=True)
housing = housing_raw.frame.copy()
housing.columns = [c.lower() for c in housing.columns]

print(f"Datasets loaded: retail {retail.shape}, monthly {monthly.shape}, credit {credit.shape}")

---
## Exercise 1 — 2×2 Grid: Basic subplot layout

**Spec:** Build a 2×2 subplot grid — the most common dashboard layout.

- Use `make_subplots(rows=2, cols=2, subplot_titles=[...])`
- Subplot titles: `['Monthly Revenue', 'Orders by Day of Week', 'Revenue by Country', 'Avg Order Value Trend']`
- Panel (1,1): Line chart of monthly Revenue — `go.Scatter`, color `'#1565C0'`
- Panel (1,2): Bar chart of orders by day of week, sorted Mon–Sun — `go.Bar`, color `'#43A047'`
- Panel (2,1): Horizontal bar of top 10 countries by Revenue, sorted ascending — `go.Bar`, color `'#FB8C00'`
- Panel (2,2): Line + markers of monthly AvgOrderValue — `go.Scatter`, color `'#8E24AA'`
- Overall title: `'Retail Operations Overview'`
- `height=700`
- Assign to `fig1`

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_orders = (
    retail.groupby('DayOfWeek')['InvoiceNo'].nunique()
    .reindex(day_order).reset_index()
    .rename(columns={'InvoiceNo':'Orders'})
)

# YOUR CODE HERE
fig1 = None

fig1.show()

In [ ]:
# --- ASSERTIONS ---
assert fig1.layout.title.text == 'Retail Operations Overview'
assert fig1.layout.height == 700
# 4 traces
assert len(fig1.data) == 4
# Subplot titles
annot_texts = [a.text for a in fig1.layout.annotations]
assert 'Monthly Revenue' in annot_texts
assert 'Revenue by Country' in annot_texts
# Correct axes assignments
axes = [(t.xaxis, t.yaxis) for t in fig1.data]
assert ('x', 'y') in axes  # panel (1,1)
assert ('x2', 'y2') in axes  # panel (1,2)
print("✓ Exercise 1 passed")

---
## Exercise 2 — Shared Axes: Synchronized Zoom

**Spec:** Build a 3-row time series panel where all panels share the same x-axis — panning/zooming one syncs all.

- Use `make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.04)`
- Row 1: Revenue line — `go.Scatter`, fill `'tozeroy'`, color `'#1565C0'`
- Row 2: Orders bar chart — `go.Bar`, color `'#43A047'`
- Row 3: AvgOrderValue line with markers — `go.Scatter`, color `'#FB8C00'`
- Y-axis titles: `'Revenue (£)'`, `'Orders'`, `'Avg Order Value (£)'`
- Only show x-axis tick labels on the bottom panel
- Row heights ratio: `row_heights=[0.5, 0.25, 0.25]`
- Title: `'Retail Time Series Dashboard (Shared X-Axis)'`
- `height=600`
- Assign to `fig2`

In [ ]:
# YOUR CODE HERE
fig2 = None

fig2.show()

In [ ]:
# --- ASSERTIONS ---
assert fig2.layout.title.text == 'Retail Time Series Dashboard (Shared X-Axis)'
assert fig2.layout.height == 600
assert len(fig2.data) == 3
# Shared x-axis: rows 1 and 2 reference xaxis3 or same xaxis
assert fig2.layout.xaxis2.matches == 'x' or fig2.layout.xaxis3.matches == 'x'
# Y-axis titles
assert 'Revenue' in fig2.layout.yaxis.title.text
assert 'Orders' in fig2.layout.yaxis2.title.text
# Top panels hide x tick labels
assert fig2.layout.xaxis.showticklabels == False
print("✓ Exercise 2 passed")

---
## Exercise 3 — Mixed Chart Types in One Figure

**Spec:** Build a finance-style chart mixing bars, lines, and a volume indicator.

Generate synthetic OHLCV data (60 days):
- Row 1 (70% height): Candlestick OHLC + 20-day SMA line
- Row 2 (30% height): Volume bars — green if close > open, red otherwise
- Shared x-axis, no range slider
- Candlestick colors: increasing `'#26A69A'`, decreasing `'#EF5350'`
- Volume bar colors: match candlestick direction
- Title: `'Price & Volume Analysis'`
- `height=550`
- Assign to `fig3`

In [ ]:
np.random.seed(7)
n = 60
dates = pd.date_range('2023-01-01', periods=n, freq='B')
close = np.cumprod(1 + np.random.normal(0.001, 0.018, n)) * 100
open_ = np.concatenate([[close[0]], close[:-1]])
high = np.maximum(open_, close) * np.random.uniform(1.001, 1.015, n)
low = np.minimum(open_, close) * np.random.uniform(0.985, 0.999, n)
volume = np.random.randint(500000, 3000000, n)
sma20 = pd.Series(close).rolling(20).mean().values
bar_colors = ['#26A69A' if c >= o else '#EF5350' for c, o in zip(close, open_)]

# YOUR CODE HERE
fig3 = None

fig3.show()

In [ ]:
# --- ASSERTIONS ---
assert fig3.layout.title.text == 'Price & Volume Analysis'
assert fig3.layout.height == 550
trace_types = [t.type for t in fig3.data]
assert 'candlestick' in trace_types
assert 'bar' in trace_types
assert 'scatter' in trace_types  # SMA line
assert fig3.layout.xaxis.rangeslider.visible == False
assert fig3.layout.xaxis2.matches == 'x'
vol_trace = [t for t in fig3.data if t.type == 'bar'][0]
assert isinstance(vol_trace.marker.color, (list, tuple)), "Volume bars must have per-bar colors"
print("✓ Exercise 3 passed")

---
## Exercise 4 — Inset Chart

**Spec:** Add a small inset chart inside a larger chart — a technique for showing a zoomed region or summary.

- Main chart: scatter of `medinc` vs `medhousval` for all 20,000+ housing rows — `go.Scatter`, opacity 0.3, size 2
- Inset chart (using `fig.add_trace` with `xref='paper'`, `yref='paper'`):
  - Show only high-income subset: `medinc > 7`
  - Position in top-right corner: `xaxis=dict(domain=[0.65, 0.98])`, `yaxis=dict(domain=[0.55, 0.95])`
  - Use `go.Histogram` of `medhousval` for high-income subset
  - Color: `'#FB8C00'`
- Add a rectangle shape highlighting the `medinc > 7` region on the main chart
- Title: `'Income vs House Value (with High-Income Inset)'`
- Assign to `fig4`

In [ ]:
high_income = housing[housing['medinc'] > 7]

# YOUR CODE HERE
fig4 = None

fig4.show()

In [ ]:
# --- ASSERTIONS ---
assert fig4.layout.title.text == 'Income vs House Value (with High-Income Inset)'
trace_types = [t.type for t in fig4.data]
assert 'scatter' in trace_types
assert 'histogram' in trace_types
# Inset axis defined
assert fig4.layout.xaxis2 is not None or hasattr(fig4.layout, 'xaxis2')
# Rectangle shape highlighting high-income region
assert len(fig4.layout.shapes) >= 1
rect = [s for s in fig4.layout.shapes if s.type == 'rect']
assert len(rect) >= 1, "Rectangle highlighting high-income region required"
print("✓ Exercise 4 passed")

---
## Exercise 5 — Unequal Column Widths & Row Heights

**Spec:** Build a dashboard with a large main panel and two smaller side panels.

Layout: 1 row × 3 cols, column widths = `[0.5, 0.25, 0.25]`

- Col 1 (wide): Scatter of `medinc` vs `medhousval`, colored by quantile of `medhousval` (4 groups)
- Col 2: Histogram of `medinc`, 30 bins, color `'#1565C0'`
- Col 3: Histogram of `medhousval`, 30 bins, color `'#43A047'`
- Col 2 and 3 share y-axis (counts)
- Title: `'Housing Market Explorer'`
- `height=450`
- Assign to `fig5`

In [ ]:
housing['value_quantile'] = pd.qcut(
    housing['medhousval'], q=4, labels=['Q1','Q2','Q3','Q4']
).astype(str)

# YOUR CODE HERE
fig5 = None

fig5.show()

In [ ]:
# --- ASSERTIONS ---
assert fig5.layout.title.text == 'Housing Market Explorer'
assert fig5.layout.height == 450
trace_types = [t.type for t in fig5.data]
assert trace_types.count('histogram') == 2
# Wide scatter panel has 4 traces (one per quantile)
scatter_traces = [t for t in fig5.data if t.type == 'scatter']
assert len(scatter_traces) == 4, "One scatter trace per value quantile"
print("✓ Exercise 5 passed")

---
## Exercise 6 — Subplot with Mixed Trace Types (Geo + Cartesian)

**Spec:** Combine a map and a bar chart in one figure.

- Use `make_subplots(rows=1, cols=2, specs=[[{'type':'geo'}, {'type':'xy'}]])`
- Left panel (geo): `go.Choropleth` of top-country revenues (use ISO codes)
- Right panel (xy): `go.Bar` of same countries, horizontal, sorted by revenue ascending
- Share the same color mapping and colorscale (`'Blues'`) between both panels
- Title: `'Revenue by Country: Map & Ranking'`
- `height=450`
- Assign to `fig6`

In [ ]:
country_iso = {
    'United Kingdom':'GBR','Germany':'DEU','France':'FRA','EIRE':'IRL',
    'Spain':'ESP','Netherlands':'NLD','Belgium':'BEL','Switzerland':'CHE',
    'Portugal':'PRT','Australia':'AUS'
}
top10_rev = (
    retail.groupby('Country')['Revenue'].sum().reset_index()
    .query('Country in @country_iso').nlargest(10,'Revenue')
)
top10_rev['ISO'] = top10_rev['Country'].map(country_iso)
top10_sorted = top10_rev.sort_values('Revenue', ascending=True)

# YOUR CODE HERE
fig6 = None

fig6.show()

In [ ]:
# --- ASSERTIONS ---
assert fig6.layout.title.text == 'Revenue by Country: Map & Ranking'
assert fig6.layout.height == 450
trace_types = [t.type for t in fig6.data]
assert 'choropleth' in trace_types
assert 'bar' in trace_types
bar_trace = [t for t in fig6.data if t.type == 'bar'][0]
assert bar_trace.orientation == 'h'
x_vals = list(bar_trace.x)
assert x_vals[-1] == max(x_vals), "Bars must be sorted ascending (highest at top)"
print("✓ Exercise 6 passed")

---
## Exercise 7 — Correlation Dashboard

**Spec:** A 3-panel correlation explorer for the housing dataset.

- Layout: `make_subplots(rows=2, cols=2, specs=[[{'colspan':2},None],[{},{}]])`
  - Top row (full width): Heatmap of correlation matrix
  - Bottom-left: Scatter of `medinc` vs `medhousval` with regression line
  - Bottom-right: Scatter of `averooms` vs `medhousval` with regression line
- Regression lines: use `numpy.polyfit` to compute, display as `go.Scatter` with `mode='lines'`
- Heatmap: masked upper triangle, `colorscale='RdBu'`, `zmid=0`
- Title: `'Housing Feature Correlation Explorer'`
- `height=650`
- Assign to `fig7`

In [ ]:
feat_cols = ['medinc','houseage','averooms','avebedrms','population','aveoccup','medhousval']
corr = housing[feat_cols].corr().round(2)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
masked_corr = corr.astype(float)
masked_corr[mask] = None

# YOUR CODE HERE
fig7 = None

fig7.show()

In [ ]:
# --- ASSERTIONS ---
assert fig7.layout.title.text == 'Housing Feature Correlation Explorer'
assert fig7.layout.height == 650
trace_types = [t.type for t in fig7.data]
assert 'heatmap' in trace_types
scatter_traces = [t for t in fig7.data if t.type == 'scatter']
# 2 scatter data + 2 regression lines = at least 4 scatter traces
assert len(scatter_traces) >= 4, "Need scatter points + regression lines for both panels"
heatmap = [t for t in fig7.data if t.type == 'heatmap'][0]
assert heatmap.zmid == 0
print("✓ Exercise 7 passed")

---
## Exercise 8 — Interpretaion: Reading a Complex Dashboard

**Spec:** This exercise tests your ability to read and interpret a multi-panel chart you did NOT build.

Run the cell below to generate a complex 4-panel dashboard. Then answer the 5 questions in the markdown cell beneath it.

In [ ]:
# --- PROVIDED CHART — DO NOT MODIFY ---
customer_rev = retail.groupby('CustomerID')['Revenue'].sum()
customer_orders = retail.groupby('CustomerID')['InvoiceNo'].nunique()
customer_df = pd.DataFrame({'Revenue': customer_rev, 'Orders': customer_orders}).reset_index()
customer_df['AvgOrderValue'] = customer_df['Revenue'] / customer_df['Orders']
customer_df['Segment'] = pd.qcut(
    customer_df['Revenue'], q=4, labels=['Low','Mid-Low','Mid-High','High']
)

fig_interp = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Customer Revenue Distribution (log scale)',
        'Orders vs Revenue by Segment',
        'Avg Order Value by Segment',
        'Revenue Concentration (Lorenz Curve)'
    ],
    height=700
)
fig_interp.add_trace(go.Histogram(x=np.log1p(customer_df['Revenue']), nbinsx=50,
                                    marker_color='#1565C0', name='Revenue Dist'), row=1, col=1)
colors_seg = {'Low':'#E53935','Mid-Low':'#FB8C00','Mid-High':'#43A047','High':'#1565C0'}
for seg, grp in customer_df.groupby('Segment'):
    fig_interp.add_trace(go.Scatter(x=grp['Orders'], y=grp['Revenue'], mode='markers',
                                     marker=dict(size=4, color=colors_seg[str(seg)], opacity=0.5),
                                     name=str(seg)), row=1, col=2)
seg_aov = customer_df.groupby('Segment')['AvgOrderValue'].median().reset_index()
fig_interp.add_trace(go.Bar(x=seg_aov['Segment'].astype(str), y=seg_aov['AvgOrderValue'],
                              marker_color=['#E53935','#FB8C00','#43A047','#1565C0'],
                              name='Median AOV'), row=2, col=1)
sorted_rev = np.sort(customer_df['Revenue'].values)
cum_rev = np.cumsum(sorted_rev) / sorted_rev.sum() * 100
cum_cust = np.arange(1, len(sorted_rev)+1) / len(sorted_rev) * 100
fig_interp.add_trace(go.Scatter(x=cum_cust, y=cum_rev, mode='lines', name='Lorenz',
                                  line=dict(color='#8E24AA', width=2)), row=2, col=2)
fig_interp.add_trace(go.Scatter(x=[0,100], y=[0,100], mode='lines', name='Perfect Equality',
                                  line=dict(color='grey', dash='dash')), row=2, col=2)
fig_interp.update_layout(title='Customer Revenue Analysis Dashboard', showlegend=False)
fig_interp.show()

**Answer these 5 questions based on the dashboard above:**

1. **Distribution shape:** Is customer revenue normally distributed on the log scale? What does this imply about the raw distribution?

2. **Segment separation:** In the Orders vs Revenue scatter (top-right), are the segments cleanly separated or overlapping? What does this suggest about using order count as a segmentation feature?

3. **Average Order Value:** Does AOV increase monotonically with segment? If not, what could explain a non-monotonic pattern?

4. **Lorenz curve:** Estimate the Gini coefficient visually. What % of revenue is held by the top 20% of customers?

5. **Business action:** Based on all 4 panels together, which customer segment would you prioritize for a retention campaign, and why?

*(Write your answers here)*

---
## Exercise 9 — EDA Dashboard Function

**Spec:** Build a reusable `eda_dashboard(df, numeric_cols, categorical_col, target_col)` function that generates a 3×2 subplot grid automatically for any dataset.

Panels (in order):
1. Histogram of `target_col`
2. Correlation heatmap of `numeric_cols`
3. Box plots of `target_col` by `categorical_col` (top 6 categories)
4. Bar chart: mean `target_col` by `categorical_col`
5. Scatter of the numeric col most correlated with `target_col` vs `target_col`
6. Lorenz curve of `target_col`

Rules:
- `height=900`, `width=1100`
- Corporate theme: `plot_bgcolor='white'`, `paper_bgcolor='#FAFAFA'`
- Each panel must have an appropriate title
- Must work on both `credit` (target: `credit_amount`) and `housing` (target: `medhousval`)

In [ ]:
def eda_dashboard(df: pd.DataFrame, numeric_cols: list,
                   categorical_col: str, target_col: str) -> go.Figure:
    """
    Automated 3×2 EDA dashboard for any dataset.
    Returns a go.Figure.
    """
    # YOUR CODE HERE
    pass

credit_numeric = ['credit_amount', 'duration', 'age']
fig9_credit = eda_dashboard(credit, credit_numeric, 'purpose', 'credit_amount')
fig9_credit.show()

housing_numeric = ['medinc', 'houseage', 'averooms', 'population', 'medhousval']
housing_cat = pd.cut(housing['houseage'], bins=5, labels=['0-10','11-20','21-30','31-40','41-52'])
housing_temp = housing.assign(age_group=housing_cat.astype(str))
fig9_housing = eda_dashboard(housing_temp, housing_numeric, 'age_group', 'medhousval')
fig9_housing.show()

In [ ]:
# --- ASSERTIONS ---
import plotly.basedatatypes
for fig, name in [(fig9_credit, 'credit'), (fig9_housing, 'housing')]:
    assert isinstance(fig, plotly.basedatatypes.BaseFigure)
    assert fig.layout.height == 900
    assert fig.layout.width == 1100
    assert len(fig.data) >= 6, f"{name} dashboard must have at least 6 traces"
    assert fig.layout.paper_bgcolor == '#FAFAFA'
    trace_types = [t.type for t in fig.data]
    assert 'histogram' in trace_types, f"{name}: missing histogram"
    assert 'heatmap' in trace_types, f"{name}: missing heatmap"
print("✓ Exercise 9 passed")

---
## Exercise 10 — Capstone: Executive Report Figure

**Spec:** Build a single publication-ready figure that tells the complete story of the retail dataset — the kind you'd present to a C-suite audience.

Layout: Custom grid using `make_subplots` with `specs` for mixed types:
```
Row 1: [KPI indicators × 3 columns (colspan each)]
Row 2: [Revenue trend (colspan=2) | Country bar (colspan=1)]
Row 3: [DOW heatmap (colspan=1) | Customer segment donut (colspan=1) | AOV distribution (colspan=1)]
```

Requirements:
- Row 1: 3 `go.Indicator` KPI tiles (Total Revenue, Total Orders, Total Customers)
- Row 2 left: Revenue + Orders dual-axis time series
- Row 2 right: Top 10 country horizontal bar
- Row 3 left: Revenue by DOW × Hour heatmap
- Row 3 middle: Customer segment donut (4 revenue quartiles)
- Row 3 right: Histogram of log(Revenue) per transaction
- Overall title: `'UK Retail Analytics — Executive Summary'`
- `height=900`, corporate styling throughout
- Assign to `fig10`

In [ ]:
# Pre-compute required data
dow_hour = (
    retail.groupby(['DayOfWeek','Hour'])['Revenue'].sum()
    .reset_index()
    .pivot(index='DayOfWeek', columns='Hour', values='Revenue')
    .reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
    .fillna(0)
)

customer_rev = retail.groupby('CustomerID')['Revenue'].sum()
seg_labels = pd.qcut(customer_rev, q=4, labels=['Low','Mid-Low','Mid-High','High'])
seg_counts = seg_labels.value_counts().sort_index()

# YOUR CODE HERE
fig10 = None

fig10.show()

In [ ]:
# --- ASSERTIONS ---
assert fig10.layout.title.text == 'UK Retail Analytics — Executive Summary'
assert fig10.layout.height == 900
trace_types = [t.type for t in fig10.data]
assert 'indicator' in trace_types, "KPI tiles missing"
assert trace_types.count('indicator') >= 3
assert 'heatmap' in trace_types, "DOW×Hour heatmap missing"
assert 'pie' in trace_types, "Customer segment donut missing"
assert 'histogram' in trace_types, "Revenue distribution missing"
assert 'bar' in trace_types or 'scatter' in trace_types, "Time series or bar missing"
assert fig10.layout.paper_bgcolor == '#FAFAFA'
print("✓ Exercise 10 passed — Executive report complete")